Earthquake Prediction & Analysis

In [1]:
import pandas as pd
import random
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
from sklearn.ensemble import RandomForestClassifier
import torch.nn as nn

In [2]:
data = []

for _ in range(500):
    latitude = random.uniform(10, 40)
    longitude = random.uniform(60, 100)
    depth = random.uniform(1, 300)
    magnitude = round(random.uniform(2.0, 8.0), 1)

    if magnitude > 5:
        earthquake = 1
    else:
        earthquake = 0

    data.append([latitude, longitude, depth, magnitude, earthquake])

df = pd.DataFrame(data, columns=[
    "latitude", "longitude", "depth", "magnitude", "earthquake"
])

df.to_csv("earthquake.csv", index=False)
print("Dataset created ")

Dataset created 


In [3]:
df = pd.read_csv("earthquake.csv")

X = df.drop("earthquake", axis=1)
y = df["earthquake"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train.values, dtype=torch.long)
y_test = torch.tensor(y_test.values, dtype=torch.long)

In [4]:
ml_model = RandomForestClassifier()
ml_model.fit(X_train.numpy(), y_train.numpy())

ml_acc = ml_model.score(X_test.numpy(), y_test.numpy())
print("ML Accuracy:", ml_acc)

ML Accuracy: 1.0


In [5]:
class EarthquakeModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 32)
        self.fc2 = nn.Linear(32, 16)
        self.out = nn.Linear(16, 2)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.out(x)

model = EarthquakeModel()

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(50):
    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print("Epoch:", epoch, "Loss:", loss.item())

Epoch: 0 Loss: 0.6816255450248718
Epoch: 10 Loss: 0.6509553790092468
Epoch: 20 Loss: 0.6185527443885803
Epoch: 30 Loss: 0.5798271298408508
Epoch: 40 Loss: 0.5303698182106018


In [6]:
with torch.no_grad():
    outputs = model(X_test)
    _, predicted = torch.max(outputs, 1)

    acc = (predicted == y_test).sum().item() / len(y_test)

print("DL Accuracy:", acc)

DL Accuracy: 0.85


In [7]:
def predict_earthquake(lat, lon, depth, magnitude):
    import pandas as pd
    import torch

    data = pd.DataFrame([[lat, lon, depth, magnitude]],
                        columns=["latitude", "longitude", "depth", "magnitude"])

    data = scaler.transform(data)
    tensor = torch.tensor(data, dtype=torch.float32)

    with torch.no_grad():
        output = model(tensor)
        _, pred = torch.max(output, 1)

    return "Earthquake Likely" if pred.item() == 1 else "No Earthquake"

In [8]:
print(predict_earthquake(28.6, 77.2, 50, 6.5))

Earthquake Likely
